# Inference

This notebook validates the saved next-category recommendation models and defines a reusable inference function.

The function returns the primary recommended category together with a ranked list of alternative categories and their predicted probabilities.

In [3]:
# -------------------------------------------------------------------
# Load Final Models And Define Inference Function
# -------------------------------------------------------------------

from pathlib import Path

import joblib
import pandas as pd

MODELS_DIR = Path("../models")

top1_model = joblib.load(
    MODELS_DIR / "next_category_top1_random_forest.joblib"
)

ranking_model = joblib.load(
    MODELS_DIR / "next_category_ranking_random_forest.joblib"
)


def predict_next_categories(feature_row: pd.DataFrame) -> dict:
    """
    Predict the next recommended household categories.

    Parameters
    ----------
    feature_row : pd.DataFrame
        A single-row DataFrame containing the same model features
        used during training.

    Returns
    -------
    dict
        Top-1 recommendation plus ranked Top-2 and Top-3 results.
    """

    # Remove identifiers not used by the trained models.
    model_input = feature_row.drop(
        columns=[
            "Property_ID",
            "Customer_ID",
            "Prediction_Time",
        ],
        errors="ignore",
    ).copy()

    # -----------------------------------------------------------
    # Top-1 model
    # -----------------------------------------------------------

    top1_category = top1_model.predict(
        model_input
    )[0]

    top1_probabilities = top1_model.predict_proba(
        model_input
    )[0]

    top1_classes = top1_model.named_steps[
        "classifier"
    ].classes_

    top1_probability = dict(
        zip(
            top1_classes,
            top1_probabilities,
        )
    )[top1_category]

    # -----------------------------------------------------------
    # Ranking model
    # -----------------------------------------------------------

    ranking_probabilities = ranking_model.predict_proba(
        model_input
    )[0]

    ranking_classes = ranking_model.named_steps[
        "classifier"
    ].classes_

    ranked_results = sorted(
        zip(
            ranking_classes,
            ranking_probabilities,
        ),
        key=lambda item: item[1],
        reverse=True,
    )

    return {
        "top_1": {
            "category": top1_category,
            "probability": round(
                float(top1_probability),
                4,
            ),
        },
        "top_2": [
            {
                "category": category,
                "probability": round(
                    float(probability),
                    4,
                ),
            }
            for category, probability in ranked_results[:2]
        ],
        "top_3": [
            {
                "category": category,
                "probability": round(
                    float(probability),
                    4,
                ),
            }
            for category, probability in ranked_results[:3]
        ],
    }


print("INFERENCE MODELS LOADED")
print("-" * 60)
print("Top-1 model loaded successfully.")
print("Ranking model loaded successfully.")

INFERENCE MODELS LOADED
------------------------------------------------------------
Top-1 model loaded successfully.
Ranking model loaded successfully.


## Recreate Holdout Test Split

The original grouped train-validation-test split is reconstructed using the same random seed so that inference can be tested on the same untouched holdout properties used during model evaluation.

In [5]:
# -------------------------------------------------------------------
# Recreate Holdout Test Split
# -------------------------------------------------------------------

from sklearn.model_selection import GroupShuffleSplit

DATA_PATH = Path("../data/processed/model_state.csv")

model_state = pd.read_csv(
    DATA_PATH,
    parse_dates=["Prediction_Time"],
)

# Separate predictors, target and property groups.
X = model_state.drop(
    columns=[
        "Next_Category_ID",
        "Next_Category",
    ]
).copy()

y = model_state["Next_Category"].copy()

groups = model_state["Property_ID"].copy()

# ---------------------------------------------------------------
# Reproduce original 70% / 15% / 15% grouped split.
# ---------------------------------------------------------------

train_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42,
)

train_idx, temp_idx = next(
    train_splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_temp = X.iloc[temp_idx].copy()
y_temp = y.iloc[temp_idx].copy()

temp_groups = groups.iloc[temp_idx].copy()

validation_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=42,
)

val_idx, test_idx = next(
    validation_splitter.split(
        X_temp,
        y_temp,
        groups=temp_groups,
    )
)

X_test = X_temp.iloc[test_idx].copy()
y_test = y_temp.iloc[test_idx].copy()

print("TEST SPLIT RECREATED")
print("-" * 60)

print(f"Test rows:       {len(X_test):,}")
print(f"Test properties: {X_test['Property_ID'].nunique():,}")

TEST SPLIT RECREATED
------------------------------------------------------------
Test rows:       55
Test properties: 36


## End-To-End Inference Test

A real unseen test observation is passed through the saved inference pipeline to verify that the persisted models produce ranked recommendations correctly.

In [6]:
# -------------------------------------------------------------------
# End-To-End Inference Test
# -------------------------------------------------------------------

# Select one unseen test observation.
sample_index = X_test.index[0]

sample_features = X_test.loc[
    [sample_index]
].copy()

actual_category = y_test.loc[
    sample_index
]

# Generate ranked recommendations.
prediction_result = predict_next_categories(
    sample_features
)

print("END-TO-END INFERENCE TEST")
print("-" * 70)

print(f"Property ID:     {sample_features['Property_ID'].iloc[0]}")
print(f"Actual category: {actual_category}")

print()

print("TOP-1 RECOMMENDATION")
print(
    f"{prediction_result['top_1']['category']} "
    f"({prediction_result['top_1']['probability']:.2%})"
)

print()

print("TOP-2 RANKING")
for rank, item in enumerate(
    prediction_result["top_2"],
    start=1,
):
    print(
        f"{rank}. {item['category']} "
        f"({item['probability']:.2%})"
    )

print()

print("TOP-3 RANKING")
for rank, item in enumerate(
    prediction_result["top_3"],
    start=1,
):
    print(
        f"{rank}. {item['category']} "
        f"({item['probability']:.2%})"
    )

END-TO-END INFERENCE TEST
----------------------------------------------------------------------
Property ID:     PRP-000040
Actual category: Other Items

TOP-1 RECOMMENDATION
Electronics (33.60%)

TOP-2 RANKING
1. Electronics (31.42%)
2. Other Items (21.99%)

TOP-3 RANKING
1. Electronics (31.42%)
2. Other Items (21.99%)
3. Clothings (15.24%)


## Package Inference Validation

The reusable prediction function is imported from the project package and tested against an unseen holdout observation.

This confirms that the persisted models and the deployment-oriented inference module work independently from the modelling notebook.

In [15]:
# -------------------------------------------------------------------
# Reload Predict Module
# -------------------------------------------------------------------

import importlib

import my_val_capstone_01.modeling.predict as predict_module

# Force Jupyter to reload the current version of predict.py
importlib.reload(predict_module)

print("PREDICT MODULE CHECK")
print("-" * 70)

print(f"Loaded from: {predict_module.__file__}")

print()
print("Available objects:")
print(
    [
        name
        for name in dir(predict_module)
        if not name.startswith("_")
    ]
)

PREDICT MODULE CHECK
----------------------------------------------------------------------
Loaded from: C:\Users\cmcjj\OneDrive - UTS\Documents\2026-2\capstone\my_val_capstone_01\my_val_capstone_01\modeling\predict.py

Available objects:
['MODELS_DIR', 'NON_MODEL_COLUMNS', 'PROCESSED_DATA_DIR', 'PROJECT_ROOT', 'Path', 'RANKING_MODEL_PATH', 'TOP1_MODEL_PATH', 'app', 'joblib', 'logger', 'main', 'pd', 'predict_next_categories', 'ranking_model', 'top1_model', 'tqdm', 'typer']


In [16]:
# -------------------------------------------------------------------
# Package Inference Validation
# -------------------------------------------------------------------

from my_val_capstone_01.modeling.predict import (
    predict_next_categories,
)

# Reuse one unseen test observation.
sample_index = X_test.index[0]

sample_features = X_test.loc[
    [sample_index]
].copy()

actual_category = y_test.loc[
    sample_index
]

# Generate prediction through the reusable package module.
package_prediction = predict_next_categories(
    sample_features
)

print("PACKAGE INFERENCE VALIDATION")
print("-" * 70)

print(
    f"Property ID:     "
    f"{sample_features['Property_ID'].iloc[0]}"
)

print(
    f"Actual category: "
    f"{actual_category}"
)

print()

print("TOP-1")
print(
    f"{package_prediction['top_1']['category']} "
    f"({package_prediction['top_1']['probability']:.2%})"
)

print()

print("TOP-2")
for rank, item in enumerate(
    package_prediction["top_2"],
    start=1,
):
    print(
        f"{rank}. {item['category']} "
        f"({item['probability']:.2%})"
    )

print()

print("TOP-3")
for rank, item in enumerate(
    package_prediction["top_3"],
    start=1,
):
    print(
        f"{rank}. {item['category']} "
        f"({item['probability']:.2%})"
    )

PACKAGE INFERENCE VALIDATION
----------------------------------------------------------------------
Property ID:     PRP-000040
Actual category: Other Items

TOP-1
Electronics (33.60%)

TOP-2
1. Electronics (31.42%)
2. Other Items (21.99%)

TOP-3
1. Electronics (31.42%)
2. Other Items (21.99%)
3. Clothings (15.24%)


## Package Inference Test

The reusable inference function is tested using an unseen holdout observation to confirm that the saved models can be loaded and executed independently from the modelling notebook.

In [17]:
# -------------------------------------------------------------------
# Package Inference Test
# -------------------------------------------------------------------

# Use the function directly from the reloaded module.
sample_index = X_test.index[0]

sample_features = X_test.loc[
    [sample_index]
].copy()

actual_category = y_test.loc[
    sample_index
]

prediction = predict_module.predict_next_categories(
    sample_features
)

print("PACKAGE INFERENCE TEST")
print("-" * 70)

print(
    f"Property ID:     "
    f"{sample_features['Property_ID'].iloc[0]}"
)

print(
    f"Actual category: "
    f"{actual_category}"
)

print()

print("TOP-1 RECOMMENDATION")
print(
    f"{prediction['top_1']['category']} "
    f"({prediction['top_1']['probability']:.2%})"
)

print()

print("TOP-2 RANKING")
for rank, item in enumerate(
    prediction["top_2"],
    start=1,
):
    print(
        f"{rank}. {item['category']} "
        f"({item['probability']:.2%})"
    )

print()

print("TOP-3 RANKING")
for rank, item in enumerate(
    prediction["top_3"],
    start=1,
):
    print(
        f"{rank}. {item['category']} "
        f"({item['probability']:.2%})"
    )

PACKAGE INFERENCE TEST
----------------------------------------------------------------------
Property ID:     PRP-000040
Actual category: Other Items

TOP-1 RECOMMENDATION
Electronics (33.60%)

TOP-2 RANKING
1. Electronics (31.42%)
2. Other Items (21.99%)

TOP-3 RANKING
1. Electronics (31.42%)
2. Other Items (21.99%)
3. Clothings (15.24%)


## Final Conclusion

This project developed a next-category recommendation approach for household contents documentation using historical customer, property, inventory, AI-analysis and video-derived information.

The modelling task was designed as a sequential recommendation problem rather than a one-time classification task. Each property could generate multiple historical prediction opportunities as its documented household state evolved. At every prediction point, only information genuinely available at that moment was used, while the target represented the next previously undocumented category observed later in the customer journey.

A key methodological focus was the prevention of data leakage. Historical household states were reconstructed using only assets already documented at each prediction timestamp, simultaneous category events were handled without inventing an artificial internal order, and train, validation and test partitions were separated by `Property_ID`. The final holdout set remained untouched during model development and hyperparameter selection.

Several modelling approaches were evaluated, including Logistic Regression, Random Forest and CatBoost. The experiments showed that more complex models could achieve stronger predictive performance but were also prone to substantial overfitting because of the relatively small number of historical prediction states.

The standard Random Forest produced the strongest Top-1 validation performance, while a regularised Random Forest provided better ranking behaviour and reduced, although did not eliminate, the generalisation gap. On the final holdout set, the Top-1 model achieved 50.91% accuracy. The ranking model achieved 70.91% Top-2 accuracy and 83.64% Top-3 accuracy.

These results suggest that the problem is particularly suitable for a ranked recommendation experience. Rather than relying exclusively on a single predicted category, the system can provide one primary suggestion together with additional relevant alternatives.

The ablation experiment also showed that the outputs of myVal's existing AI system contributed useful predictive information. Removing AI-derived features reduced validation Macro F1 from 0.3676 to 0.3324. Validation permutation importance further showed that AI-derived signals, such as suggestion acceptance, category agreement and estimated values, contributed alongside the customer's existing documented category profile.

Performance remained uneven across categories. Electronics and Furniture were predicted more reliably, while categories with limited representation, including Clothings, Jewellery and Personal Items, and Vehicles, were more difficult to identify. This class imbalance, together with the relatively small modelling dataset, represents an important limitation of the current experiments.

Overall, the project demonstrates that historical household state and existing AI outputs can be combined to generate useful next-category recommendations. The strongest evidence was found in the ranked recommendation setting, where the correct category was frequently present among the first two or three suggestions.

However, the current application and dataset are still at an early stage of maturity. The number of historical recommendation opportunities remains relatively small, and several target categories are underrepresented. As a result, the current findings should be interpreted as evidence of technical feasibility rather than as proof of a fully mature production model.

Further customer interaction data, additional household histories and a more balanced representation of categories would provide a stronger basis for future training and evaluation. As the application grows and more real user behaviour becomes available, the recommendation system could be retrained and reassessed to improve generalisation, reduce class imbalance and better capture real customer documentation patterns.

### Final Deliverables

The final workflow produced:

- a leakage-controlled historical modelling dataset;
- grouped train, validation and holdout test partitions;
- multiple baseline, tuned and regularised modelling experiments;
- an AI-feature ablation study;
- feature-importance and permutation-importance analysis;
- separate persisted models for Top-1 and ranked recommendations;
- a reusable inference function capable of returning Top-1, Top-2 and Top-3 category recommendations.

The deployment layer was intentionally left outside the scope of the current implementation.